# FI-2010: Tuned DeepLOB and Best LSTM

Previous run got DeepLOB at 75% (vs published 80.5%). The training curves showed overfitting
after epoch 25 with lr=0.01. This notebook tries:

1. **DeepLOB + LR scheduler**: Start at 0.001, cosine anneal over 200 epochs, weight decay
2. **DeepLOB + step decay**: lr=0.01 decayed by 0.1 every 30 epochs
3. **DeepLOB + warmup**: Linear warmup for 5 epochs then cosine decay
4. **Best LSTM**: Take the improved LSTM (68.4%) and tune it further

In [1]:
import os, glob, subprocess, time, copy, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR, StepLR, LambdaLR
from sklearn.metrics import classification_report, f1_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
print("Setup complete")

Device: cuda
GPU: Tesla T4
Setup complete


In [2]:
print("Downloading FI-2010...")
subprocess.run(['wget', '-q',
    'https://raw.githubusercontent.com/zcakhaa/DeepLOB-Deep-Convolutional-Neural-Networks-for-Limit-Order-Books/master/data/data.zip',
    '-O', '/kaggle/working/data.zip'], check=True)
subprocess.run(['unzip', '-q', '-o', '/kaggle/working/data.zip', '-d', '/kaggle/working/'], check=True)
DATA_DIR = '/kaggle/working'
print("Done.")

Done.


In [3]:
def prepare_x(data):
    return data[:40, :].T.astype(np.float32)

def get_label(data):
    return data[-5:, :].T.astype(int) - 1

def data_classification(X, Y, T=100):
    N = X.shape[0]
    samples = N - T + 1
    X_seq = np.zeros((samples, T, X.shape[1]), dtype=np.float32)
    for i in range(samples):
        X_seq[i] = X[i:i+T]
    return X_seq, Y[T-1:]

test_files = sorted(glob.glob(os.path.join(DATA_DIR, "Test_Dst_NoAuction*.txt")))
dec_train = np.loadtxt(os.path.join(DATA_DIR, 'Train_Dst_NoAuction_DecPre_CF_7.txt'))
dec_test = np.hstack([np.loadtxt(tf) for tf in test_files])

train_lob, train_label = prepare_x(dec_train), get_label(dec_train)
test_lob, test_label = prepare_x(dec_test), get_label(dec_test)

T, HORIZON = 100, 3
X_train_seq, y_train_seq = data_classification(train_lob, train_label, T)
X_test_seq, y_test_seq = data_classification(test_lob, test_label, T)
y_train_all, y_test = y_train_seq[:, HORIZON], y_test_seq[:, HORIZON]

val_split = int(len(X_train_seq) * 0.8)
X_val, y_val = X_train_seq[val_split:], y_train_all[val_split:]
X_train, y_train = X_train_seq[:val_split], y_train_all[:val_split]
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test_seq.shape}")

class LOBDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

BS = 64
train_ld = DataLoader(LOBDataset(X_train, y_train), BS, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_ld = DataLoader(LOBDataset(X_val, y_val), BS, shuffle=False, num_workers=2, pin_memory=True)
test_ld = DataLoader(LOBDataset(X_test_seq, y_test), BS, shuffle=False, num_workers=2, pin_memory=True)
print(f"Batches: train={len(train_ld)}, val={len(val_ld)}, test={len(test_ld)}")

Train: (203720, 100, 40) | Val: (50931, 100, 40) | Test: (139488, 100, 40)


Batches: train=3183, val=796, test=2180


In [4]:
class DeepLOB(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, (1, 2), stride=(1, 2)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, (4, 1)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, (4, 1)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 32, (1, 2), stride=(1, 2)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, (4, 1)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, (4, 1)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 32, (1, 10)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, (4, 1)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, (4, 1)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
        )
        self.inp1 = nn.Sequential(
            nn.Conv2d(32, 64, (1, 1)), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, (3, 1), padding=(1, 0)), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
        )
        self.inp2 = nn.Sequential(
            nn.Conv2d(32, 64, (1, 1)), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, (5, 1), padding=(2, 0)), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
        )
        self.inp3 = nn.Sequential(
            nn.MaxPool2d((3, 1), stride=(1, 1), padding=(1, 0)),
            nn.Conv2d(32, 64, (1, 1)), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
        )
        self.lstm = nn.LSTM(192, 64, 1, batch_first=True)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = torch.cat((self.inp1(x), self.inp2(x), self.inp3(x)), dim=1)
        x = x.squeeze(-1).permute(0, 2, 1)
        x, _ = self.lstm(x)
        return self.fc(x[:, -1, :])

class BestLSTM(nn.Module):
    """Best effort LSTM: 3 layers, 256 hidden, dropout, trained properly."""
    def __init__(self, input_size=40, hidden_size=256, n_layers=3, output_size=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, n_layers,
                           batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :]))

print(f"DeepLOB params: {sum(p.numel() for p in DeepLOB().parameters()):,}")
print(f"BestLSTM params: {sum(p.numel() for p in BestLSTM().parameters()):,}")

DeepLOB params: 143,907
BestLSTM params: 1,358,595


In [5]:
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total += x.size(0)
    return correct / total

def full_test(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            all_preds.append(model(x.to(device)).argmax(1).cpu())
            all_labels.append(y)
    preds = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    print(f"Test accuracy: {acc*100:.2f}%")
    print(f"Test F1 (weighted): {f1*100:.2f}%")
    print(classification_report(labels, preds, target_names=['Down','Stationary','Up']))
    return acc, f1

def train_with_scheduler(model, train_ld, val_ld, test_ld,
                         lr, weight_decay, epochs, patience,
                         scheduler_type, device, name, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, eps=1.0, weight_decay=weight_decay)

    if scheduler_type == 'cosine':
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    elif scheduler_type == 'step':
        scheduler = StepLR(optimizer, step_size=30, gamma=0.1)
    elif scheduler_type == 'warmup_cosine':
        warmup_epochs = 5
        def lr_lambda(epoch):
            if epoch < warmup_epochs:
                return epoch / warmup_epochs
            progress = (epoch - warmup_epochs) / (epochs - warmup_epochs)
            return 0.5 * (1 + math.cos(math.pi * progress))
        scheduler = LambdaLR(optimizer, lr_lambda)
    else:
        scheduler = None

    criterion = nn.CrossEntropyLoss()
    best_val_acc, best_state, no_improve = 0, None, 0
    start = time.time()

    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"lr={lr}, wd={weight_decay}, scheduler={scheduler_type}, epochs={epochs}")
    print(f"{'='*60}")

    for epoch in range(epochs):
        model.train()
        train_correct, train_total = 0, 0
        for x, y in train_ld:
            x, y = x.to(device), y.to(device)
            loss = criterion(model(x), y)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_correct += (model(x).detach().argmax(1) == y).sum().item()
            train_total += x.size(0)

        if scheduler:
            scheduler.step()

        train_acc = train_correct / train_total
        val_acc = evaluate(model, val_ld, device)
        current_lr = optimizer.param_groups[0]['lr']

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
            marker = ' *'
        else:
            no_improve += 1
            marker = ''

        if epoch % 10 == 0 or no_improve == 0:
            print(f"Ep {epoch:>3} | Tr: {train_acc:.4f} | Va: {val_acc:.4f} | "
                  f"Best: {best_val_acc:.4f} | lr: {current_lr:.6f} | {time.time()-start:.0f}s{marker}")

        if no_improve >= patience:
            print(f"Early stop at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    elapsed = time.time() - start
    print(f"\nBest val: {best_val_acc*100:.2f}% | Time: {elapsed/60:.1f}min")
    acc, f1 = full_test(model, test_ld, device)
    return model, acc, f1

## Run 1: DeepLOB + Cosine Annealing (lr=0.001, 200 epochs)

The previous run overfitted with lr=0.01. Lower LR with cosine decay should train
slower but generalise better.

In [6]:
results = {}

m, acc, f1 = train_with_scheduler(
    DeepLOB(), train_ld, val_ld, test_ld,
    lr=0.001, weight_decay=1e-5, epochs=200, patience=40,
    scheduler_type='cosine', device=device,
    name='DeepLOB + Cosine (lr=0.001, wd=1e-5, 200ep)'
)
results['DeepLOB Cosine'] = {'acc': acc, 'f1': f1}


DeepLOB + Cosine (lr=0.001, wd=1e-5, 200ep)
lr=0.001, wd=1e-05, scheduler=cosine, epochs=200


Ep   0 | Tr: 0.3661 | Va: 0.3661 | Best: 0.3661 | lr: 0.001000 | 184s *


Ep   2 | Tr: 0.3980 | Va: 0.3664 | Best: 0.3664 | lr: 0.000999 | 569s *


Ep   4 | Tr: 0.4142 | Va: 0.3717 | Best: 0.3717 | lr: 0.000998 | 957s *


Ep   5 | Tr: 0.4206 | Va: 0.3832 | Best: 0.3832 | lr: 0.000998 | 1150s *


...

Ep 180 | Tr: 0.7203 | Va: 0.5847 | Best: 0.5883 | lr: 0.000023 | 35113s


Ep 187 | Tr: 0.7204 | Va: 0.5883 | Best: 0.5883 | lr: 0.000010 | 36471s *


Ep 190 | Tr: 0.7202 | Va: 0.5885 | Best: 0.5885 | lr: 0.000006 | 37053s *



Best val: 58.85% | Time: 646.6min


Test accuracy: 72.95%
Test F1 (weighted): 72.61%
              precision    recall  f1-score   support

        Down       0.65      0.63      0.64     38408
  Stationary       0.80      0.86      0.83     65996
          Up       0.66      0.59      0.62     35084

    accuracy                           0.73    139488
   macro avg       0.70      0.69      0.70    139488
weighted avg       0.72      0.73      0.73    139488



## Run 2: DeepLOB + Step Decay (lr=0.01, decay 0.1 every 30 epochs)

Keeps the aggressive lr=0.01 from the paper but decays it before overfitting sets in.

In [7]:
m, acc, f1 = train_with_scheduler(
    DeepLOB(), train_ld, val_ld, test_ld,
    lr=0.01, weight_decay=1e-5, epochs=150, patience=40,
    scheduler_type='step', device=device,
    name='DeepLOB + StepLR (lr=0.01, decay@30, wd=1e-5)'
)
results['DeepLOB Step'] = {'acc': acc, 'f1': f1}


DeepLOB + StepLR (lr=0.01, decay@30, wd=1e-5)
lr=0.01, wd=1e-05, scheduler=step, epochs=150


Ep   0 | Tr: 0.4003 | Va: 0.3617 | Best: 0.3617 | lr: 0.010000 | 194s *


Ep   1 | Tr: 0.4983 | Va: 0.4528 | Best: 0.4528 | lr: 0.010000 | 388s *


Ep   2 | Tr: 0.5998 | Va: 0.5234 | Best: 0.5234 | lr: 0.010000 | 581s *


Ep   3 | Tr: 0.6678 | Va: 0.5477 | Best: 0.5477 | lr: 0.010000 | 775s *


...

Ep  15 | Tr: 0.7648 | Va: 0.6187 | Best: 0.6187 | lr: 0.010000 | 3102s *


Ep  16 | Tr: 0.7674 | Va: 0.6270 | Best: 0.6270 | lr: 0.010000 | 3296s *


Ep  17 | Tr: 0.7713 | Va: 0.6282 | Best: 0.6282 | lr: 0.010000 | 3491s *


Ep  18 | Tr: 0.7733 | Va: 0.6328 | Best: 0.6328 | lr: 0.010000 | 3685s *


Ep  20 | Tr: 0.7793 | Va: 0.6319 | Best: 0.6328 | lr: 0.010000 | 4073s


## Run 3: DeepLOB + Warmup + Cosine (lr=0.003)

Linear warmup for 5 epochs avoids the initial instability, then cosine decay.

In [ ]:
m, acc, f1 = train_with_scheduler(
    DeepLOB(), train_ld, val_ld, test_ld,
    lr=0.003, weight_decay=1e-5, epochs=200, patience=40,
    scheduler_type='warmup_cosine', device=device,
    name='DeepLOB + Warmup+Cosine (lr=0.003, wd=1e-5)'
)
results['DeepLOB Warmup'] = {'acc': acc, 'f1': f1}

## Run 4: Best LSTM (3-layer, 256h, Adam lr=0.001, cosine)

Push the LSTM as far as it can go. 3 layers, 256 hidden, dropout 0.3, cosine schedule.

In [ ]:
m, acc, f1 = train_with_scheduler(
    BestLSTM(hidden_size=256, n_layers=3, dropout=0.3), train_ld, val_ld, test_ld,
    lr=0.001, weight_decay=1e-5, epochs=100, patience=30,
    scheduler_type='cosine', device=device,
    name='Best LSTM (3L, 256h, dropout=0.3, cosine)'
)
results['Best LSTM 3L'] = {'acc': acc, 'f1': f1}

## Run 5: Best LSTM (2-layer, 512h)

Wider instead of deeper.

In [ ]:
m, acc, f1 = train_with_scheduler(
    BestLSTM(hidden_size=512, n_layers=2, dropout=0.3), train_ld, val_ld, test_ld,
    lr=0.001, weight_decay=1e-5, epochs=100, patience=30,
    scheduler_type='cosine', device=device,
    name='Best LSTM (2L, 512h, dropout=0.3, cosine)'
)
results['Best LSTM 512'] = {'acc': acc, 'f1': f1}

## Summary

In [ ]:
print("=" * 70)
print("FI-2010 TUNED RESULTS (k=50, Setup 2)")
print("=" * 70)
print(f"{'Model':<50} {'Acc':>8} {'F1':>8}")
print("-" * 68)

for name, r in results.items():
    print(f"  {name:<48} {r['acc']*100:>6.2f}% {r['f1']*100:>6.2f}%")

print("-" * 68)
print(f"  {'Previous: DeepLOB (lr=0.01, no scheduler)':<48} {'75.00%':>8} {'74.70%':>8}")
print(f"  {'Previous: Improved LSTM (256h)':<48} {'68.41%':>8} {'68.29%':>8}")
print(f"  {'Previous: Dissertation LSTM (128h)':<48} {'63.54%':>8} {'63.80%':>8}")
print(f"  {'Published DeepLOB (Zhang 2019)':<48} {'80.51%':>8} {'80.35%':>8}")
print(f"  {'Dissertation RSNN BNTT+LT':<48} {'59.15%':>8} {'---':>8}")